# Validacion y Exploracion — Dataset Colegios de Chile

**7,673 establecimientos · 345 comunas · 16 regiones · 6 tablas Parquet**

Este notebook explora el dataset generado por el pipeline ETL.
No ejecuta extraccion — solo analisis sobre los Parquet en `data/processed/latest/`.

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

# VS Code a veces no carga user site-packages. Forzarlo.
py_ver = f'python{sys.version_info.major}.{sys.version_info.minor}'
user_site = Path.home() / '.local' / 'lib' / py_ver / 'site-packages'
if user_site.exists() and str(user_site) not in sys.path:
    sys.path.insert(0, str(user_site))

import polars as pl
import duckdb

# Encontrar raiz del proyecto (buscar plan.md hacia arriba)
PROJECT = Path(os.getcwd())
for _ in range(5):
    if (PROJECT / 'plan.md').exists():
        break
    PROJECT = PROJECT.parent
if not (PROJECT / 'plan.md').exists():
    PROJECT = Path('/home/rasgdev/projects/colegios-chile')

DATA = PROJECT / 'data' / 'processed' / 'latest'
print(f"Proyecto: {PROJECT}")
print(f"Data dir: {DATA.resolve()}")
for f in sorted(DATA.glob("*.parquet")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:40s} {size_mb:.1f} MB")

## 2. Carga de datos

In [ ]:
est = pl.read_parquet(DATA / "establecimientos.parquet")
sed = pl.read_parquet(DATA / "sedes.parquet")
cur = pl.read_parquet(DATA / "cursos.parquet")
act = pl.read_parquet(DATA / "actividades.parquet")
ind = pl.read_parquet(DATA / "indicadores.parquet")
img = pl.read_parquet(DATA / "imagenes.parquet")

print(f"{"Tabla":<25} {"Filas":>8} {"Columnas":>10}")
print("-" * 45)
for name, df in [
    ("establecimientos", est),
    ("sedes", sed),
    ("cursos", cur),
    ("actividades", act),
    ("indicadores", ind),
    ("imagenes", img),
]:
    print(f"{name:<25} {len(df):>8,} {len(df.columns):>10}")
total = len(est) + len(sed) + len(cur) + len(act) + len(ind) + len(img)
print(f"\nTotal combinado: {total:,} registros")

## 3. Establecimientos

Cada fila es un colegio. Veamos distribuciones clave.

In [ ]:
print("=== Dependencia ===")
print(est["dependencia"].value_counts().sort("count", descending=True))

print("\n=== Alumnos matriculados ===")
print(est["alumnos_matriculados"].describe())

print("\n=== Colegios con 0 alumnos ===")
cero = est.filter(pl.col('alumnos_matriculados') == 0)
print(f'RBDs con 0 alumnos: {len(cero)}')
if len(cero) > 0:
    print(cero.select("rbd", "nombre", "dependencia").head(10))

In [ ]:
print("=== Top 10 colegios por matricula ===")
top = (
    est.select("rbd", "nombre", "dependencia", "alumnos_matriculados")
    .sort("alumnos_matriculados", descending=True)
    .head(10)
)
print(top)

print("\n=== Colegios sin director registrado ===")
sin_director = est.filter(pl.col("director").is_null())
pct = len(sin_director) / len(est) * 100
print(f"Sin director: {len(sin_director)} de {len(est)} ({pct:.1f}%)")

## 4. Distribucion geografica

In [ ]:
print("=== Top 10 regiones por cantidad de sedes ===")
top_regiones = (
    sed.group_by("region")
    .agg(pl.len().alias("sedes"))
    .sort("sedes", descending=True)
)
print(top_regiones.head(10))

print("\n=== Top 10 comunas por cantidad de colegios ===")
comunas_por_rbd = (
    est.join(
        sed.select("rbd", "comuna").unique(subset=["rbd"]),
        on="rbd", how="inner",
    )
    .group_by("comuna")
    .agg(pl.len().alias("colegios"))
    .sort("colegios", descending=True)
)
print(comunas_por_rbd.head(10))

## 5. Dependencia vs Matricula

Que tipo de colegio concentra mas alumnos?

In [ ]:
print("=== Matricula promedio por dependencia ===")
resumen = (
    est.group_by("dependencia")
    .agg(
        pl.col("alumnos_matriculados").mean().alias("promedio_alumnos"),
        pl.col("alumnos_matriculados").median().alias("mediana_alumnos"),
        pl.len().alias("colegios"),
        pl.col("alumnos_matriculados").sum().alias("total_alumnos"),
    )
    .sort("colegios", descending=True)
)
print(resumen)

print("\n=== Colegios con internado ===")
con_internado = (
    est.filter(pl.col('internado') == True)
    .select("rbd", "nombre", "dependencia", "alumnos_matriculados")
    .sort("alumnos_matriculados", descending=True)
    .head(10)
)
print(con_internado)

## 6. Cursos y Copago

La tabla `cursos` tiene 77,540 filas — una por cada nivel en cada sede.

In [ ]:
print("=== Distribucion de copago mensual ===")
cur_con_copago = cur.filter(
    pl.col("copago_valor").is_not_null() & (pl.col("copago_valor") > 0)
)
print(cur_con_copago["copago_valor"].describe())

print("\n=== Top 10 copagos mas altos ===")
top_copago = (
    cur_con_copago
    .join(est.select("rbd", "nombre", "dependencia"), on="rbd")
    .select("rbd", "nombre", "dependencia", "glosa_nivel", "copago_valor")
    .sort("copago_valor", descending=True)
    .head(10)
)
print(top_copago)

In [ ]:
print("=== Distribucion de vacantes por nivel ===")
vacantes = (
    cur.group_by("glosa_nivel")
    .agg(
        pl.col("cupos_totales").sum().alias("total_cupos"),
        pl.col("vacantes_rango_inferior").sum().alias("vacantes_min"),
        pl.col("vacantes_rango_superior").sum().alias("vacantes_max"),
    )
    .sort("total_cupos", descending=True)
)
print(vacantes.head(15))

## 7. Actividades Extraprogramaticas

206,041 registros clasificados por tipo.

In [ ]:
print("=== Actividades por tipo ===")
print(act["tipo"].value_counts().sort("count", descending=True))

print("\n=== Actividades mas comunes ===")
top_actividades = (
    act.filter(pl.col("nombre").is_not_null())
    .group_by("nombre")
    .agg(pl.len().alias("colegios"))
    .sort("colegios", descending=True)
    .head(15)
)
print(top_actividades)

## 8. Indicadores (SIMCE y Desempeno)

In [ ]:
print("=== Tipos de indicador ===")
print(ind["tipo_indicador"].value_counts().sort("count", descending=True))

print("\n=== SIMCE Lenguaje — top 20 ===")
simce_leng = ind.filter(
    (pl.col("tipo_indicador") == "SIMCE")
    & (pl.col("nombre_indicador") == "Lenguaje")
)
top_simce = (
    simce_leng
    .join(est.select("rbd", "nombre", "dependencia"), on="rbd")
    .select("nombre", "dependencia", "puntaje", "titulo_indicador")
    .sort("puntaje", descending=True)
    .head(20)
)
print(top_simce)

print("\n=== Estadisticas SIMCE ===")
for materia in ["Lenguaje", "Matematica"]:
    subset = ind.filter(
        (pl.col("nombre_indicador") == materia)
        & (pl.col("puntaje").is_not_null())
    )
    if len(subset) > 0:
        print(f"\n{materia}:")
        print(f"  Colegios con dato: {len(subset)}")
        media = subset['puntaje'].mean()
        print(f"  Puntaje promedio:  {media:.1f}")
        maximo = subset['puntaje'].max()
        print(f"  Puntaje maximo:    {maximo}")

## 9. Validacion de integridad (DuckDB)

Las mismas 6 queries de `make validate`, ahora interactivas.

In [ ]:
con = duckdb.connect(":memory:")
data_dir = str(DATA.resolve())

checks = {
    "Sedes huerfanas": f"SELECT COUNT(*) FROM '{data_dir}/sedes.parquet' s LEFT JOIN '{data_dir}/establecimientos.parquet' e ON s.rbd = e.rbd WHERE e.rbd IS NULL",
    "Cursos sin sede": f"SELECT COUNT(*) FROM '{data_dir}/cursos.parquet' c LEFT JOIN '{data_dir}/sedes.parquet' s ON c.rbd = s.rbd AND c.codigo_sede = s.codigo_sede WHERE s.rbd IS NULL",
    "RBDs duplicados": f"SELECT COUNT(*) FROM (SELECT rbd, COUNT(*) n FROM '{data_dir}/establecimientos.parquet' GROUP BY rbd HAVING n > 1)",
    "Est. sin sedes": f"SELECT COUNT(*) FROM '{data_dir}/establecimientos.parquet' e LEFT JOIN '{data_dir}/sedes.parquet' s ON e.rbd = s.rbd WHERE s.rbd IS NULL",
    "Nulos clave (est.)": f"SELECT COUNT(*) FROM '{data_dir}/establecimientos.parquet' WHERE rbd IS NULL OR nombre IS NULL",
    "Nulos clave (sedes)": f"SELECT COUNT(*) FROM '{data_dir}/sedes.parquet' WHERE rbd IS NULL OR codigo_sede IS NULL",
}

for nombre, query in checks.items():
    result = con.execute(query).fetchone()
    count = result[0]
    if count == 0:
        print(f"[      OK] {nombre}")
    else:
        print(f"[FALLO {count}] {nombre}")

con.close()

## 10. Hallazgos y conclusiones

Ejecuta todas las celdas y anota aca lo que encontraste:

- **Distribucion por dependencia**: que tipo de colegio predomina?
- **Matricula**: hay concentracion en pocos colegios grandes o esta distribuida?
- **Copagos**: rango de valores? outliers?
- **Geografia**: la Region Metropolitana concentra mas del 30%?
- **SIMCE**: correlacion visible entre dependencia y puntajes?
- **Calidad de datos**: los 2 nulos reportados, son errores o datos faltantes legitimos?

---
*Dataset generado con colegios-chile · Datos: API publica MINEDUC (SAE)*